In [ ]:
LOGISTIC REGRESSION BUILT IN 

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


# Step 1: Load Data
path = r"C:\Users\Girija S.H\.cache\kagglehub\datasets\prachi13\customer-analytics\versions\1"
train_path = os.path.join(path, "Train.csv")
df = pd.read_csv(train_path)

# Drop ID column
df = df.drop(columns=["ID"])

# Separate features and target
X = df.drop(columns=["Reached.on.Time_Y.N"])
y = df["Reached.on.Time_Y.N"]

# One-hot encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# Scale numerical features
scaler = StandardScaler()
X[X.columns] = scaler.fit_transform(X)

# ------------------------------
# Step 2: Train-Test Split
# ------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 3: Logistic Regression + Hyperparameter Tuning
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],  # Regularization strength
    "solver": ["liblinear", "saga"],  # Solvers that handle imbalance well
    "penalty": ["l1", "l2"]  # Regularization type
}

log_reg = LogisticRegression(class_weight="balanced", max_iter=5000, random_state=42)

grid_search = GridSearchCV(log_reg, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

# Step 4: Evaluation
y_pred = best_model.predict(X_test)

print("Best Parameters:", grid_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


In [ ]:
LOGISTIC REGRESSION FROM SCRATCH

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Step 1: Load and Preprocess
path = r"C:\Users\Girija S.H\.cache\kagglehub\datasets\prachi13\customer-analytics\versions\1"
train_path = os.path.join(path, "Train.csv")
df = pd.read_csv(train_path)

# Drop ID
df = df.drop(columns=["ID"])

# Separate features and target
X = df.drop(columns=["Reached.on.Time_Y.N"])
y = df["Reached.on.Time_Y.N"]

# One-hot encode categoricals
X = pd.get_dummies(X, drop_first=True)

# Scale numerical features
scaler = StandardScaler()
X[X.columns] = scaler.fit_transform(X)

# Convert to numpy arrays
X = X.values
y = y.values.reshape(-1, 1)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 2: Define Logistic Regression from Scratch
class LogisticRegressionScratch:
    def __init__(self, lr=0.01, n_iters=1000):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.bias = None

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros((n_features, 1))
        self.bias = 0

        # Gradient Descent
        for _ in range(self.n_iters):
            linear_model = np.dot(X, self.weights) + self.bias
            y_predicted = self.sigmoid(linear_model)

            # Gradients
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)

            # Update weights
            self.weights -= self.lr * dw
            self.bias -= self.lr * db

    def predict_proba(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        return self.sigmoid(linear_model)

    def predict(self, X):
        y_pred_proba = self.predict_proba(X)
        return np.where(y_pred_proba >= 0.5, 1, 0)


# Step 3: Train & Evaluate
model = LogisticRegressionScratch(lr=0.01, n_iters=5000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Accuracy
accuracy = np.mean(y_pred == y_test)
print("Accuracy (Scratch Logistic Regression):", accuracy)
